# Phase 4 — controlled TOR3-PAGI ReID contribution comparison

**REQUIREMENT:** run all cells in a fresh Kaggle GPU session with the original traffic dataset and byte-identical Phase 2 retained cache attached. This notebook never loads the detector or runs detection inference. It runs two independent, sequential BoT-SORT passes over the same 10,282 cached frames: ReID off, then the dedicated official `yolo26n-reid.onnx` encoder on. Decoder passes are deliberately separate to isolate tracker/encoder state without storing an entire video in RAM; the pinned tracker uses its own per-instance ID iterator.

**PROPOSAL — untuned candidate, documented before implementation:** see `docs/PHASE_4_TASK.md`. Ultralytics 8.4.175, lap 0.5.13 and all Phase 3 motion/lifecycle/association thresholds remain fixed; GMC stays `none`. The encoder was trained on MSMT17-v1, a person ReID dataset. Transfer to vehicle appearance is UNKNOWN. Successful execution and changed IDs/costs do not establish identity improvement. There is no ground truth.

The native `model='auto'` path is excluded because detector features were not retained. There is no alternate encoder or CPU fallback. CUDA availability, actual ONNX CUDA execution, finite nonzero embeddings, their path into association and saved temporal evidence are verified explicitly. Raw classes remain unmapped. No geometry, crossing, counting, night preprocessing, GMC/stabilization, UI or multi-GPU orchestration is implemented. Phase 5 is not started.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import json, os, platform, subprocess, sys, time, uuid

VIDEO_PATH = Path('/kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-PAGI.mp4')
DETECTION_REPORT_PATH = None  # Optional explicit Path; otherwise discover one exact parent run.
FRAME_DETECTIONS_PATH = None  # Optional explicit Path; otherwise resolve the parent report artifact.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/reid_comparison')
PARENT_RUN_ID = '20261011T052950_360560Z_ed399fb8'
EXPECTED_VIDEO_SHA256 = '606cfdd26c2d0a6892f5680d6a35d7e4d372efd76dd6bc56e2e4cbed7a74754c'
EXPECTED_CACHE_SHA256 = '033a2815d3abbd23327936a1b9d948b2b9b7d312796516e5ee46aec5685be345'
EXPECTED_FRAME_COUNT = 10282
PACKAGE_VERSION = '8.4.175'
LAP_VERSION = '0.5.13'
TRACKER_OVERRIDES = {'with_reid': False, 'gmc_method': 'none'}
ONNX_VERSION = '1.20.1'
ORT_VERSION = '1.24.4'
REID_DEVICE = 'cuda:0'
ENCODER_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n-reid.onnx'
EXPECTED_ENCODER_SHA256 = '8529c383197ae4c468eda535d1b165f8b4162cf17bf5fbcff49c7cb6455bc0bb'
EXPECTED_ENCODER_SIZE = 9873245
ENCODER_PATH = None  # Optional existing exact asset; otherwise download the verified official release.
VARIANTS = ['reid_off', 'reid_on']
CUDA_PROFILE_REQUIRED = True
BURSTS = [(0, 29), (2555, 2584), (5125, 5154), (7696, 7725), (10252, 10281)]
REPRESENTATIVE_INDICES = [0, 2570, 5140, 7711, 10281]
FOCUS_INDICES = list(range(7711, 7719))
FOCUS_CROP_XYXY = (500, 250, 1280, 720)  # Display-only audit crop; never used for association/eligibility.
RED_FOCUS_INDICES = list(range(5125, 5131))
RED_FOCUS_CROP_XYXY = (0, 250, 750, 720)  # Display only.
SEED = 0

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ_') + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'samples').mkdir()
(RUN_DIR / 'temporal').mkdir()
os.environ['YOLO_CONFIG_DIR'] = str(RUN_DIR / '.ultralytics_settings')
Path(os.environ['YOLO_CONFIG_DIR']).mkdir()
REPORT = {
    'phase': 4, 'run_id': RUN_ID, 'started_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'initialized', 'claim_type': 'EXPERIMENT_RESULT_ON_EXECUTION',
    'scope': 'controlled cached ReID off/on; CPU association, dedicated ONNX encoder cuda:0; no detector or counting',
    'configuration': {'video_path': str(VIDEO_PATH), 'parent_run_id': PARENT_RUN_ID,
                      'variants': VARIANTS, 'reid_device': REID_DEVICE,
                      'tracker_overrides': TRACKER_OVERRIDES, 'seed': SEED,
                      'baseline_status': 'PROPOSAL: published, untuned tracker defaults with explicit isolation overrides'},
    'stages': {}, 'frames_processed': 0, 'variants': {},
    'manual_review_status': 'pending_actual_temporal_evidence_review',
}

def write_report():
    (RUN_DIR / 'reid_report.json').write_text(json.dumps(REPORT, indent=2, allow_nan=False)+'\n')
    lines = ['# Phase 4 execution report', '', f"Status: `{REPORT['status']}`",
             f"Run: `{RUN_ID}`", f"Parent: `{PARENT_RUN_ID}`", '',
             'Same retained detections, independent BoT-SORT off/on variants; no detector rerun or geometry/counting.',
             f"Frames processed: {REPORT['frames_processed']}",
             'Unique tracker IDs are lifecycle diagnostics, not physical vehicle or traffic totals.',
             f"Visual review: {REPORT['manual_review_status']}"]
    if 'failure' in REPORT: lines.extend(['', 'Failure: '+REPORT['failure']['message']])
    if 'diagnostics' in REPORT:
        lines.extend(['', '## Observed software/lifecycle diagnostics', '',
                      '```json', json.dumps(REPORT['diagnostics'], indent=2), '```'])
    (RUN_DIR / 'reid_report.md').write_text('\n'.join(lines)+'\n')

from contextlib import contextmanager
@contextmanager
def stage(name):
    REPORT['current_stage'] = name
    started = time.perf_counter()
    write_report()
    try:
        yield
    except Exception as exc:
        REPORT['status'] = 'failed_partial_execution'
        REPORT['failure'] = {'stage': name, 'type': type(exc).__name__, 'message': str(exc)}
        raise
    finally:
        REPORT['stages'][name] = {'wall_seconds': time.perf_counter()-started}
        write_report()

write_report()
print('Run directory:', RUN_DIR)


In [ ]:
with stage('dependency_installation'):
    wanted = {'ultralytics': PACKAGE_VERSION, 'lap': LAP_VERSION, 'onnx': ONNX_VERSION, 'onnxruntime-gpu': ORT_VERSION}
    missing = []
    for package, version in wanted.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            if package.replace('-', '_') in sys.modules or (package == 'onnxruntime-gpu' and 'onnxruntime' in sys.modules):
                raise RuntimeError(f'{package} already imported with a different version; restart this session.')
            missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *missing], check=True)
    for package, version in wanted.items():
        assert metadata.version(package) == version, (package, metadata.version(package))


In [ ]:
import hashlib, math, statistics, random
from collections import Counter
import cv2
import numpy as np
import torch
import ultralytics
from ultralytics.engine.results import Boxes
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import TrackState, BaseTrack
from ultralytics.trackers.utils import matching
import onnxruntime as ort
import urllib.request
import resource
from ultralytics.utils import YAML, IterableSimpleNamespace
from PIL import Image, ImageDraw

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

with stage('environment_and_verified_tracker_source'):
    package_root = Path(ultralytics.__file__).resolve().parent
    EXPECTED_SOURCE_HASHES = {'trackers/bot_sort.py': 'b157cbc33f75f1132df530f04361824bc39a30d4051ce6df1cd5c388e49b3117', 'trackers/byte_tracker.py': 'fb45d70e952f99984085788bc09fc0c60a18b14bc4329b28e32bc2c70ac37401', 'trackers/basetrack.py': '9ef094b9a54ff2f52c9118b5bd1a2f84ba331aa557277f54186de11728c2f7e8', 'trackers/utils/matching.py': '318bf81030558863a4ec6effea3c9d063a297f299367e379a58b89c24e3f421f', 'trackers/utils/gmc.py': 'd797f701a22347001d780b0dc4d0f2f5cabdfefb895c36bbab084ceb2dfc4000', 'trackers/utils/stracks.py': '9534294af850c3a6755c3a5a1ba9b2b068c5b79a11166c8547652944e2632f58', 'trackers/utils/reid.py': '75c54a4f8fe9b1e6946d61b530fbb790fdfcc27df120fa71cc7759c11e9f0475', 'cfg/trackers/botsort.yaml': '8cd88930d3be5557be0777f384eecff0a36a7011850fe613fc0a68178bfd5b01', 'engine/results.py': '871e5b3eed1558c3c048d99058ddf28a7927c1317d020981e7c0836f18e00ee3'}
    EXPECTED_SOURCE_HASHES.update({'nn/backends/onnx.py': '8a42c894bb34381ffa2636d2bf8704c5ea5f2f80db6de296b278c6c6528b6e5e', 'nn/autobackend.py': '450d5d5301e441896f8422cd333ce35c414c2e7ac39a88af01fd5edbd2e552dd', 'utils/plotting.py': 'a21122beb75642c535ba5f20f81e51cf9fd9c22c097120d1e787a7286c468ff0'})
    source_checks = []
    for relative_path, expected_hash in EXPECTED_SOURCE_HASHES.items():
        path = package_root / relative_path
        actual_hash = sha256_file(path)
        require(actual_hash == expected_hash, f'Pinned tracker source mismatch: {relative_path}')
        source_checks.append({'path': relative_path, 'sha256': actual_hash, 'verified': True})
    cuda_devices = []
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            device = torch.cuda.get_device_properties(index)
            cuda_devices.append({'index': index, 'name': device.name, 'total_memory_bytes': device.total_memory})
    try:
        smi = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                              '--format=csv,noheader'], capture_output=True, text=True, timeout=15)
        smi_observation = {'returncode': smi.returncode, 'stdout': smi.stdout.strip(), 'stderr': smi.stderr.strip()}
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        smi_observation = {'unavailable': type(exc).__name__}
    REPORT['environment'] = {
        'python': sys.version, 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
        'package_versions': {name: metadata.version(name) for name in
                             ['ultralytics', 'lap', 'torch', 'numpy', 'Pillow', 'onnx', 'onnxruntime-gpu']},
        'opencv_version': cv2.__version__, 'cuda_available': torch.cuda.is_available(),
        'visible_cuda_device_count': len(cuda_devices), 'cuda_devices': cuda_devices,
        'nvidia_smi': smi_observation, 'association_device': 'cpu',
        'detector_inference_executed': False, 'reid_inference_executed': False,
        'ort_available_providers': ort.get_available_providers(),
        'requested_encoder_device': REID_DEVICE,
        'exact_cross_runtime_repeatability_claimed': False,
    }
    REPORT['tracker_source_verification'] = source_checks
    require(REID_DEVICE == 'cuda:0' and torch.cuda.is_available(), 'This explicit GPU experiment requires cuda:0; no CPU fallback.')
    require('CUDAExecutionProvider' in ort.get_available_providers(), 'ONNX Runtime CUDA provider unavailable; no CPU fallback.')
    print(json.dumps(REPORT['environment'], indent=2))


In [ ]:
def discover_parent_report(explicit_path, search_root, expected_run_id):
    if explicit_path is not None:
        path = Path(explicit_path)
        data = json.loads(path.read_text())
        require(data.get('run_id') == expected_run_id, 'Explicit parent report has the wrong run_id.')
        return path, data
    matches = []
    for path in Path(search_root).rglob('detection_report.json'):
        try:
            data = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if data.get('run_id') == expected_run_id:
            matches.append((path, data))
    require(len(matches) == 1,
            f'Expected one mounted parent report for {expected_run_id}; found {len(matches)}. '
            'Attach retained Phase 2 outputs/cache, or set DETECTION_REPORT_PATH explicitly. No YOLO rerun is allowed.')
    return matches[0]

def verify_input_hash(path, expected):
    require(Path(path).is_file(), f'Input file not found: {path}')
    actual = sha256_file(path)
    require(actual == expected, f'SHA-256 mismatch for {path}: {actual}')
    return actual

def validate_detection_row(row, expected_index, video_sha, width, height, class_names):
    require(type(row.get('frame_index')) is int and row['frame_index'] == expected_index,
            f'Out-of-order/missing/extra cached frame: expected {expected_index}.')
    require(row.get('video_sha256') == video_sha, f'Wrong video identity at frame {expected_index}.')
    require(row.get('width') == width and row.get('height') == height,
            f'Wrong dimensions at frame {expected_index}.')
    require(isinstance(row.get('detections'), list), 'Detections must be a list.')
    for index, det in enumerate(row['detections']):
        cls = det.get('class_id')
        require(type(cls) is int and str(cls) in class_names, f'Invalid raw class at {expected_index}/{index}.')
        require(det.get('class_name') == class_names[str(cls)], 'Class name does not match the retained model mapping.')
        confidence = det.get('confidence')
        require(isinstance(confidence, (int, float)) and math.isfinite(confidence) and 0 <= confidence <= 1,
                'Invalid detection confidence.')
        box = det.get('bbox_xyxy_pixels')
        require(isinstance(box, list) and len(box) == 4 and all(isinstance(v, (int, float)) and math.isfinite(v) for v in box),
                'Invalid raw bounding box.')
        x1, y1, x2, y2 = box
        require(0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height, 'Raw detection box must be positive and in-frame.')
    return row

with stage('retained_input_identity_and_cache_validation'):
    DETECTION_REPORT_PATH, PARENT_REPORT = discover_parent_report(DETECTION_REPORT_PATH, INPUT_ROOT, PARENT_RUN_ID)
    require(PARENT_REPORT.get('phase') == 2, 'Parent must be an actual Phase 2 report.')
    require(PARENT_REPORT['input']['sha256'] == EXPECTED_VIDEO_SHA256, 'Parent video hash does not match selected input.')
    require(PARENT_REPORT['artifacts']['frame_detections_sha256'] == EXPECTED_CACHE_SHA256, 'Parent cache hash does not match.')
    require(PARENT_REPORT['execution']['frames_processed'] == EXPECTED_FRAME_COUNT, 'Parent frame total differs from scope.')
    FRAME_DETECTIONS_PATH = (Path(FRAME_DETECTIONS_PATH) if FRAME_DETECTIONS_PATH is not None
                             else DETECTION_REPORT_PATH.parent / PARENT_REPORT['artifacts']['frame_detections'])
    require(FRAME_DETECTIONS_PATH.suffix == '.jsonl', 'Use the retained uncompressed JSONL (not a silently substituted cache).')
    video_hash = verify_input_hash(VIDEO_PATH, EXPECTED_VIDEO_SHA256)
    cache_hash = verify_input_hash(FRAME_DETECTIONS_PATH, EXPECTED_CACHE_SHA256)
    CLASS_NAMES = PARENT_REPORT['model']['class_names']
    WIDTH = int(PARENT_REPORT['input']['opencv_metadata']['width'])
    HEIGHT = int(PARENT_REPORT['input']['opencv_metadata']['height'])
    FPS = float(PARENT_REPORT['input']['opencv_metadata']['fps'])
    require(math.isfinite(FPS) and FPS > 0, 'Parent FPS unavailable; buffer semantics cannot assume 30 FPS.')
    CACHE_ROWS = []
    with FRAME_DETECTIONS_PATH.open() as handle:
        for index, line in enumerate(handle):
            require(index < EXPECTED_FRAME_COUNT, 'Unexpected extra detection row.')
            row = json.loads(line)
            CACHE_ROWS.append(validate_detection_row(row, index, video_hash, WIDTH, HEIGHT, CLASS_NAMES))
    require(len(CACHE_ROWS) == EXPECTED_FRAME_COUNT, 'Cache ended before all retained frames were available.')
    REPORT['input'] = {
        'video_path': str(VIDEO_PATH), 'video_sha256': video_hash, 'video_size_bytes': VIDEO_PATH.stat().st_size,
        'detection_report_path': str(DETECTION_REPORT_PATH), 'detection_report_sha256': sha256_file(DETECTION_REPORT_PATH),
        'frame_detections_path': str(FRAME_DETECTIONS_PATH), 'frame_detections_sha256': cache_hash,
        'parent_run_id': PARENT_RUN_ID, 'parent_status': PARENT_REPORT['status'],
        'expected_and_validated_rows': len(CACHE_ROWS), 'width': WIDTH, 'height': HEIGHT,
        'fps_from_parent_opencv': FPS, 'parent_input_metadata': PARENT_REPORT['input'],
        'parent_detector_configuration': PARENT_REPORT['configuration'],
        'parent_detector_environment': PARENT_REPORT['environment'],
        'parent_detector_weights': PARENT_REPORT['weights'],
        'timestamp_semantics': 'Tracking uses zero-based frame order; index/FPS is approximate, backend timestamps are unverified PTS.',
    }
    print('Validated retained cache:', FRAME_DETECTIONS_PATH, len(CACHE_ROWS), 'rows; no detection executed.')


In [ ]:
with stage('tracker_configuration'):
    YAML_PATH = package_root / 'cfg/trackers/botsort.yaml'
    PUBLISHED_DEFAULTS = YAML.load(YAML_PATH)
    TRACKER_CONFIG = dict(PUBLISHED_DEFAULTS, **TRACKER_OVERRIDES)
    require(TRACKER_CONFIG['tracker_type'] == 'botsort', 'Tracker substitution is not allowed.')
    require(TRACKER_CONFIG['with_reid'] is False and TRACKER_CONFIG['gmc_method'] == 'none',
            'Off baseline must preserve Phase 3 isolation overrides.')
    REPORT['tracker'] = {
        'class': 'ultralytics.trackers.bot_sort.BOTSORT', 'package_version': PACKAGE_VERSION,
        'published_yaml_sha256': sha256_file(YAML_PATH), 'published_defaults': PUBLISHED_DEFAULTS,
        'explicit_overrides': TRACKER_OVERRIDES, 'effective_configuration': TRACKER_CONFIG,
        'association_device': 'cpu', 'class_gate_in_verified_source': False,
        'class_semantics': 'Raw class/score follow the latest matched detection; no product mapping or stabilization.',
        'lost_buffer_semantics': 'Pinned constructor uses track_buffer directly as update-call/frame count, no FPS rescaling.',
        'max_frames_lost': TRACKER_CONFIG['track_buffer'],
        'buffer_seconds_estimate_from_parent_fps': TRACKER_CONFIG['track_buffer']/FPS,
        'internal_source_constants': {'second_stage_iou_cost_limit': 0.5, 'unconfirmed_cost_limit': 0.7,
                                      'tracked_lost_duplicate_iou_distance_ceiling': 0.15},
        'reid_architecture_status': 'Target architecture; this Phase 4 tests one external-encoder configuration.',
    }
    print(json.dumps(REPORT['tracker'], indent=2))


with stage('verified_encoder_asset'):
    if ENCODER_PATH is None:
        ENCODER_PATH = RUN_DIR / 'assets' / 'yolo26n-reid.onnx'
        ENCODER_PATH.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(ENCODER_URL, ENCODER_PATH)
    ENCODER_PATH = Path(ENCODER_PATH)
    require(ENCODER_PATH.stat().st_size == EXPECTED_ENCODER_SIZE, 'Dedicated encoder size mismatch.')
    verify_input_hash(ENCODER_PATH, EXPECTED_ENCODER_SHA256)
    REPORT['encoder_candidate'] = {
        'identifier': 'yolo26n-reid.onnx', 'path': str(ENCODER_PATH), 'source_url': ENCODER_URL,
        'sha256': sha256_file(ENCODER_PATH), 'size_bytes': ENCODER_PATH.stat().st_size,
        'candidate_status': 'PROPOSAL: one untuned candidate; not a final accepted vehicle ReID solution',
        'documented_training_domain': 'MSMT17-v1 person ReID; vehicle transfer UNKNOWN',
        'embedding_dimension_expected': 512, 'device': REID_DEVICE,
        'crop_size': 224, 'crop_size_provenance': 'Pinned ReID default with dynamic ONNX height/width; metadata export 448 is not applied.',
        'preprocessing': 'Exact pinned wrapper: save_one_box gain1.02/pad10 default then channel reversal, float32 CHW /255, bilinear224 align_cornersFalse. No custom channel fix or enhancement.',
        'feature_postprocessing': 'Pinned BOTrack L2 normalization and normalized EMA alpha0.9.',
    }
    VARIANT_CONFIGS = {
        'reid_off': dict(TRACKER_CONFIG),
        'reid_on': dict(TRACKER_CONFIG, with_reid=True, model=str(ENCODER_PATH), device=REID_DEVICE),
    }
    invariant_keys = [key for key in TRACKER_CONFIG if key not in ['with_reid', 'model', 'device']]
    require(all(VARIANT_CONFIGS['reid_off'][key] == VARIANT_CONFIGS['reid_on'][key]
                for key in invariant_keys), 'Comparison changes motion/lifecycle/association settings.')
    REPORT['variant_configurations'] = VARIANT_CONFIGS


## Evidence and comparison semantics

Records reuse the Phase 3 lifecycle/trajectory schema. Every observation links to the retained source frame and original detection index; histories contain assigned detections, including unconfirmed ones, rather than invented motion measurements. Predicted/lost states remain separate. Centers are diagnostic values, not a selected geometry anchor. Direction/counting state remain `None`.

Each variant has its own records, summary and ID scope. Numeric IDs between off/on may diverge: compare source detections and visible physical components, not matching ID numbers. ID/gap/class-change totals are software diagnostics, not vehicle counts, identity accuracy or an improvement score.

The encoder observer records actual finite nonzero 512-dimensional output, feature hashes/norms and timing. The association observer returns the unchanged parent result and verifies it against the pinned IoU/score/appearance formula; changed costs are mechanism evidence, not evidence of better physical identity. Profiling is enabled on the actual backend session, and retained kernel events must demonstrate CUDA execution. Merely listing a provider is insufficient.


In [ ]:
STATE_NAMES = {TrackState.New: 'new', TrackState.Tracked: 'tracked',
               TrackState.Lost: 'lost', TrackState.Removed: 'removed'}

class EvidenceBOTSORT(BOTSORT):
    """Observe library object references; preserve the parent association algorithm."""
    def update(self, results, img=None, feats=None, **kwargs):
        self.before_objects = list(self.tracked_stracks) + list(self.lost_stracks)
        self.created_objects = []
        return super().update(results, img=img, feats=feats, **kwargs)

    def _init_new_tracks(self, u_detection, detections, activated, refind=None):
        previous_length = len(activated)
        super()._init_new_tracks(u_detection, detections, activated, refind)
        self.created_objects.extend(activated[previous_length:])

def detection_boxes(row):
    matrix = np.asarray([det['bbox_xyxy_pixels'] + [det['confidence'], det['class_id']]
                         for det in row['detections']], dtype=np.float32).reshape(-1, 6)
    return Boxes(matrix, orig_shape=(row['height'], row['width']))

def validate_emitted(outputs, row):
    result = np.asarray(outputs, dtype=np.float32)
    if result.size == 0:
        result = np.empty((0, 8), dtype=np.float32)
    require(result.ndim == 2 and result.shape[1] == 8, 'Unexpected BoT-SORT output shape.')
    require(np.isfinite(result).all(), 'Nonfinite emitted tracker values.')
    ids, indices = set(), set()
    for output in result:
        x1, y1, x2, y2, track_id_float, score, cls_float, index_float = output.tolist()
        require(x2 > x1 and y2 > y1, 'Degenerate emitted Kalman box.')
        track_id, index, cls = int(track_id_float), int(index_float), int(cls_float)
        require(track_id_float == track_id and track_id > 0 and track_id not in ids, 'Invalid/duplicate emitted ID.')
        require(index_float == index and 0 <= index < len(row['detections']) and index not in indices,
                'Invalid/duplicate original detection linkage.')
        source = row['detections'][index]
        require(cls_float == cls and cls == source['class_id'], 'Emitted class is not latest linked raw class.')
        require(math.isclose(score, source['confidence'], abs_tol=1e-6), 'Emitted score provenance mismatch.')
        ids.add(track_id); indices.add(index)
    return result

def pool_signature(tracker):
    def signature(track):
        return (track.track_id, track.state, track.is_activated, track.frame_id, track.start_frame,
                int(track.idx), int(track.cls), float(track.score), tuple(track.xyxy.tolist()))
    return {name: [signature(track) for track in getattr(tracker, name, [])]
            for name in ['tracked_stracks', 'lost_stracks', 'removed_stracks', 'removed_stracks_frame']}

def frame_evidence(tracker, outputs, row, registry, previous_states):
    frame_index = row['frame_index']
    current_frame_id = frame_index + 1
    require(tracker.frame_id == current_frame_id, 'Tracker update count must equal cached frame index+1.')
    emitted_by_id = {int(out[4]): out for out in outputs}
    tracked = {t.track_id for t in tracker.tracked_stracks}
    lost = {t.track_id for t in tracker.lost_stracks}
    removed = {t.track_id for t in getattr(tracker, 'removed_stracks_frame', [])}
    objects = {t.track_id: t for t in tracker.before_objects + tracker.created_objects +
               tracker.tracked_stracks + tracker.lost_stracks + getattr(tracker, 'removed_stracks_frame', [])}
    snapshots, transitions, observations = [], [], []
    current_states = {}
    for track_id, track in sorted(objects.items()):
        membership = [name for name, pool in [('tracked', tracked), ('lost', lost), ('removed_this_frame', removed)]
                      if track_id in pool]
        retained = bool(membership)
        state = STATE_NAMES[track.state] if retained else 'not_retained'
        state_key = (state, bool(track.is_activated))
        current_states[track_id] = state_key
        entry = registry.setdefault(track_id, {
            'track_id': track_id, 'first_seen_frame': None, 'last_seen_frame': None,
            'detection_count': 0, 'emitted_observation_count': 0, 'missing_frame_count': 0,
            'class_id': None, 'class_name': None, 'confidence': None, 'bbox': None, 'center': None,
            'trajectory': [], 'gap_events': [], 'class_changes': [], 'lifecycle': [],
            'direction': None, 'counting_state': None,
        })
        observed = track.frame_id == current_frame_id
        if observed:
            detection_index = int(track.idx)
            require(0 <= detection_index < len(row['detections']), 'Lifecycle detection linkage out of range.')
            det = row['detections'][detection_index]
            require(int(track.cls) == det['class_id'] and math.isclose(float(track.score), det['confidence'], abs_tol=1e-6),
                    'Lifecycle class/score not backed by current raw detection.')
            require(entry['last_seen_frame'] is None or entry['last_seen_frame'] < frame_index,
                    'Per-ID history is not strictly ordered.')
            if entry['last_seen_frame'] is not None:
                gap = frame_index-entry['last_seen_frame']-1
                if gap:
                    entry['gap_events'].append({'previous_observed_frame': entry['last_seen_frame'],
                                                 'next_observed_frame': frame_index, 'missing_frames': gap})
                if entry['class_id'] != det['class_id']:
                    entry['class_changes'].append({'frame_index': frame_index, 'previous_class_id': entry['class_id'],
                                                   'new_class_id': det['class_id'], 'detection_index': detection_index})
            box = list(det['bbox_xyxy_pixels'])
            center = [(box[0]+box[2])/2, (box[1]+box[3])/2]
            observation = {
                'track_id': track_id, 'frame_index': frame_index, 'original_detection_index': detection_index,
                'raw_bbox': box, 'bbox': track.xyxy.tolist(), 'center': center,
                'class_id': det['class_id'], 'class_name': det['class_name'], 'confidence': det['confidence'],
                'timestamp_estimate_from_index_fps_seconds': frame_index/FPS,
                'timestamp_parent_backend_ms': row.get('timestamp_backend_ms'),
                'emitted': track_id in emitted_by_id, 'is_activated': bool(track.is_activated),
                'measurement_provenance': 'retained_phase_2_detection',
            }
            entry['trajectory'].append(observation)
            observations.append(observation)
            if entry['first_seen_frame'] is None: entry['first_seen_frame'] = frame_index
            entry['last_seen_frame'] = frame_index
            entry['detection_count'] += 1
            entry['emitted_observation_count'] += int(observation['emitted'])
            for field in ['class_id', 'class_name', 'confidence', 'bbox', 'center']:
                entry[field] = observation[field]
        entry['missing_frame_count'] = frame_index-entry['last_seen_frame'] if entry['last_seen_frame'] is not None else 0
        entry['last_lifecycle_frame'] = frame_index
        entry['final_recorded_state'] = state
        estimated_box = track.xyxy.tolist()
        require(all(math.isfinite(value) for value in estimated_box) and estimated_box[2] > estimated_box[0] and
                estimated_box[3] > estimated_box[1], 'Invalid finite/positive lifecycle Kalman estimate.')
        snapshot = {
            'track_id': track_id, 'state': state, 'library_state': STATE_NAMES[track.state],
            'pool_membership': membership, 'is_activated': bool(track.is_activated),
            'observed_this_frame': observed, 'emitted': track_id in emitted_by_id,
            'bbox_kalman_estimate': track.xyxy.tolist(),
            'bbox_provenance': 'filtered_current_detection' if observed else 'tracker_estimate_not_current_detection',
            'latest_observed_frame': track.frame_id-1, 'missing_frame_count': entry['missing_frame_count'],
        }
        snapshots.append(snapshot)
        old_state = previous_states.get(track_id)
        if old_state != state_key:
            event = {'track_id': track_id, 'frame_index': frame_index,
                     'previous_state': list(old_state) if old_state else None, 'state': state,
                     'is_activated': bool(track.is_activated), 'pool_membership': membership}
            transitions.append(event); entry['lifecycle'].append(event)
    for track_id in list(previous_states):
        if track_id not in current_states and previous_states[track_id][0] not in ('removed', 'not_retained'):
            event = {'track_id': track_id, 'frame_index': frame_index,
                     'previous_state': list(previous_states[track_id]), 'state': 'not_retained',
                     'is_activated': previous_states[track_id][1], 'pool_membership': []}
            transitions.append(event); registry[track_id]['lifecycle'].append(event)
            registry[track_id]['final_recorded_state'] = 'not_retained'
            previous_states[track_id] = ('not_retained', previous_states[track_id][1])
    previous_states.update(current_states)
    observed_map = {obs['track_id']: obs for obs in observations}
    require(set(emitted_by_id) <= set(observed_map), 'An emitted track lacks a current detector observation.')
    emitted = []
    for track_id, output in emitted_by_id.items():
        obs = dict(observed_map[track_id])
        obs['bbox'] = output[:4].tolist()
        emitted.append(obs)
    return {
        'video_sha256': EXPECTED_VIDEO_SHA256, 'parent_run_id': PARENT_RUN_ID, 'frame_index': frame_index,
        'width': row['width'], 'height': row['height'],
        'timestamp_estimate_from_index_fps_seconds': frame_index/FPS,
        'timestamp_parent_backend_ms': row.get('timestamp_backend_ms'),
        'raw_detections': row['detections'], 'emitted_tracks': emitted,
        'assigned_observations_including_unconfirmed': observations,
        'lifecycle_snapshots': snapshots, 'lifecycle_transitions': transitions,
    }


In [ ]:
class EncoderEvidence:
    """Read-only call observer wrapping the actual upstream external encoder."""
    def __init__(self, encoder):
        self.encoder = encoder
        self.calls = []; self.total_calls = 0; self.total_features = 0
        self.total_wall_seconds = 0.0
    def __call__(self, img, dets):
        started = time.perf_counter()
        features = self.encoder(img, dets)
        wall = time.perf_counter()-started
        require(len(features) == len(dets), 'Encoder feature/detection cardinality mismatch.')
        norms, dimensions, digests = [], [], []
        for feature in features:
            require(feature is not None, 'Empty ReID crop/feature: this run cannot claim complete appearance integration.')
            feature = np.asarray(feature)
            require(feature.shape == (512,) and np.isfinite(feature).all(), 'Unexpected/nonfinite encoder feature.')
            norm = float(np.linalg.norm(feature))
            require(math.isfinite(norm) and norm > 1e-12, 'Zero-norm appearance feature.')
            norms.append(norm); dimensions.append(int(feature.size))
            digests.append(hashlib.sha256(feature.tobytes()).hexdigest())
        self.calls.append({'feature_count': len(features), 'dimensions': sorted(set(dimensions)),
                           'feature_l2_norms': norms, 'feature_sha256': digests,
                           'encoder_wall_seconds': wall})
        self.total_calls += 1; self.total_features += len(features); self.total_wall_seconds += wall
        return features

class AppearanceEvidenceBOTSORT(EvidenceBOTSORT):
    """Observe association costs while returning the exact pinned parent result."""
    def __init__(self, args):
        self.cost_calls = []; self.cost_totals = Counter()
        super().__init__(args)
        self.encoder_evidence = EncoderEvidence(self.encoder) if self.encoder is not None else None
        if self.encoder_evidence is not None: self.encoder = self.encoder_evidence
    def get_dists(self, tracks, detections):
        started = time.perf_counter()
        actual = super().get_dists(tracks, detections)
        parent_wall = time.perf_counter()-started
        iou = matching.iou_distance(tracks, detections)
        motion = matching.fuse_score(iou.copy(), detections) if self.args.fuse_score else iou.copy()
        reference = motion.copy(); features_present = False; eligible_pairs = 0
        appearance_raw = None; appearance_gated = None
        if self.args.with_reid and self.encoder is not None:
            if len(tracks) and len(detections):
                require(all(t.smooth_feat is not None and np.isfinite(t.smooth_feat).all() for t in tracks),
                        'Track smooth features did not reach appearance association.')
                require(all(d.curr_feat is not None and np.isfinite(d.curr_feat).all() for d in detections),
                        'Detection features did not reach appearance association.')
                features_present = True
            appearance_raw = matching.embedding_distance(tracks, detections) / 2.0
            appearance_gated = appearance_raw.copy()
            eligible = (appearance_raw <= (1-self.appearance_thresh)) & (iou <= (1-self.proximity_thresh))
            eligible_pairs = int(eligible.sum())
            appearance_gated[appearance_gated > (1-self.appearance_thresh)] = 1.0
            appearance_gated[iou > (1-self.proximity_thresh)] = 1.0
            reference = np.minimum(motion, appearance_gated)
        require(np.array_equal(actual, reference), 'Read-only cost observer disagrees with pinned parent formula.')
        difference = np.abs(actual-motion)
        changed_pairs = int(np.count_nonzero(difference > 1e-12))
        record = {'frame_index': self.frame_id-1, 'track_ids': [t.track_id for t in tracks],
                  'detection_indices': [int(d.idx) for d in detections],
                  'matrix_shape': list(actual.shape), 'candidate_pairs': int(actual.size),
                  'features_present_in_nonempty_association': features_present,
                  'appearance_eligible_pairs': eligible_pairs, 'appearance_changed_cost_pairs': changed_pairs,
                  'max_cost_reduction': float(difference.max()) if difference.size else 0.0,
                  'parent_get_dists_wall_seconds': parent_wall, 'parent_formula_exact_match': True}
        if any(start <= self.frame_id-1 <= end for start, end in BURSTS):
            record.update(motion_costs=motion.tolist(), actual_costs=actual.tolist(),
                          appearance_raw_half_cosine=None if appearance_raw is None else appearance_raw.tolist(),
                          appearance_gated_costs=None if appearance_gated is None else appearance_gated.tolist())
        self.cost_calls.append(record)
        self.cost_totals.update(calls=1, candidate_pairs=int(actual.size),
                                nonempty_feature_association_calls=int(features_present),
                                appearance_eligible_pairs=eligible_pairs,
                                appearance_changed_cost_pairs=changed_pairs,
                                calls_with_changed_costs=int(changed_pairs > 0))
        return actual

from contextlib import contextmanager
@contextmanager
def profiled_ort_session_creation(prefix):
    """Enable diagnostics when the upstream backend constructs its actual session."""
    original = ort.InferenceSession
    def create(*args, **kwargs):
        args = list(args)
        options = (args[1] if len(args) > 1 else kwargs.get('sess_options')) or ort.SessionOptions()
        options.enable_profiling = True
        options.profile_file_prefix = str(prefix)
        if len(args) > 1:
            args[1] = options
            kwargs.pop('sess_options', None)
        else:
            kwargs['sess_options'] = options
        session = original(*args, **kwargs)
        session.disable_fallback()  # No runtime provider retry after a CUDA failure.
        return session
    ort.InferenceSession = create
    try: yield
    finally: ort.InferenceSession = original

def cuda_profile_evidence(session, variant_dir):
    profile_path = Path(session.end_profiling())
    require(profile_path.is_file(), 'Actual ONNX runtime profile not written.')
    profile = json.loads(profile_path.read_text())
    providers = Counter(event.get('args', {}).get('provider') for event in profile
                        if event.get('cat') == 'Node' and event.get('args', {}).get('provider'))
    cuda_nodes = providers.get('CUDAExecutionProvider', 0)
    require(cuda_nodes > 0, 'No actual CUDA kernel events; provider list alone cannot establish GPU execution.')
    cuda_examples = [{'name': event.get('name'), 'provider': event.get('args', {}).get('provider'),
                      'op_name': event.get('args', {}).get('op_name'), 'duration_us': event.get('dur')}
                     for event in profile if event.get('cat') == 'Node' and
                     event.get('args', {}).get('provider') == 'CUDAExecutionProvider'][:8]
    record = {'profile': str(profile_path.relative_to(RUN_DIR)), 'profile_sha256': sha256_file(profile_path),
              'provider_kernel_event_counts': dict(providers), 'cuda_kernel_event_examples': cuda_examples,
              'actual_cuda_execution_confirmed': True,
              'cpu_partition_note': 'CPU-assigned graph nodes, if present, are reported; no whole-session CUDA fallback is permitted.'}
    (variant_dir/'encoder_provider_evidence.json').write_text(json.dumps(record, indent=2)+'\n')
    return record


In [ ]:
with stage('cached_adapter_and_observer_validation'):
    baseline = BOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    observed_tracker = EvidenceBOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    registry, states = {}, {}
    for row in CACHE_ROWS[:200]:
        baseline_output = baseline.update(detection_boxes(row), img=None)
        observed_output = observed_tracker.update(detection_boxes(row), img=None)
        require(np.array_equal(baseline_output, observed_output), 'Evidence observer altered actual tracker output.')
        require(pool_signature(baseline) == pool_signature(observed_tracker), 'Evidence observer altered tracker lifecycle.')
        normalized = validate_emitted(observed_output, row)
        frame_evidence(observed_tracker, normalized, row, registry, states)
    empty = dict(CACHE_ROWS[0], frame_index=200, detections=[])
    result = observed_tracker.update(detection_boxes(empty), img=None)
    require(observed_tracker.frame_id == 201 and np.asarray(result).size == 0, 'Empty frame must age tracks without emitting observations.')
    fresh = EvidenceBOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    fresh_output = validate_emitted(fresh.update(detection_boxes(CACHE_ROWS[0]), img=None), CACHE_ROWS[0])
    require(len(fresh_output) and int(fresh_output[:, 4].min()) == 1, 'A new video tracker must begin a fresh ID sequence.')
    REPORT['adapter_validation'] = {
        'actual_cached_rows_checked': 200, 'output_and_lifecycle_equivalence': 'passed',
        'empty_update_aging': 'passed', 'fresh_id_reset': 'passed', 'detector_inference_executed': False,
        'scope': 'CPU cached adapter smoke before controlled Phase 4 off/on sequences; not a rerun of Phase 2.',
    }
    del baseline, observed_tracker, fresh, registry, states
    print(json.dumps(REPORT['adapter_validation'], indent=2))


In [ ]:
def color_for_id(track_id):
    return tuple(int(v) for v in np.random.default_rng(track_id).integers(80, 256, 3))

def render_tracks(frame, evidence):
    image = frame.copy()
    assigned = {obs['original_detection_index'] for obs in evidence['assigned_observations_including_unconfirmed']}
    for index, det in enumerate(evidence['raw_detections']):
        if index not in assigned:
            box = np.rint(det['bbox_xyxy_pixels']).astype(int)
            cv2.rectangle(image, tuple(box[:2]), tuple(box[2:]), (160, 160, 160), 1)
    for obs in evidence['assigned_observations_including_unconfirmed']:
        x1, y1, x2, y2 = np.rint(obs['bbox']).astype(int)
        color = color_for_id(obs['track_id']) if obs['emitted'] else (0, 150, 255)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2 if obs['emitted'] else 1)
        label = f"ID {obs['track_id']} {obs['class_name']} {obs['confidence']:.2f} d{obs['original_detection_index']}"
        if not obs['emitted']: label += ' UNCONF/NOT-EMITTED'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        tx = max(0, min(x1, image.shape[1]-tw-2)); ty = max(th+28, min(y1, image.shape[0]-2))
        cv2.rectangle(image, (tx, ty-th-4), (min(tx+tw+2, image.shape[1]-1), ty+3), (0, 0, 0), -1)
        cv2.putText(image, label, (tx+1, ty-1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    cv2.rectangle(image, (0, 0), (image.shape[1]-1, 25), (0, 0, 0), -1)
    cv2.putText(image, f"frame {evidence['frame_index']} | IDs {len(evidence['emitted_tracks'])} | cached BoT-SORT | {evidence.get('variant', 'unknown')} | GMC none",
                (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return image


In [ ]:
def variant_diagnostics(registry, rows_verified, emitted_ids, emitted_count, empty_count, no_emitted_count):
    return {'serialized_rows_verified': rows_verified, 'captured_tracker_ids_including_unconfirmed': len(registry),
            'unique_emitted_tracker_ids': len(emitted_ids), 'emitted_track_observations': emitted_count,
            'frames_with_zero_cached_detections': empty_count, 'frames_with_no_emitted_tracks': no_emitted_count,
            'ids_never_emitted': sum(e['emitted_observation_count'] == 0 for e in registry.values()),
            'within_id_gap_events': sum(len(e['gap_events']) for e in registry.values()),
            'within_id_raw_class_changes': sum(len(e['class_changes']) for e in registry.values()),
            'lifecycle_transition_events': sum(len(e['lifecycle']) for e in registry.values()),
            'id_switch_count': None, 'id_switch_count_status': 'Unknown; needs physical identity review.',
            'traffic_count': None, 'traffic_count_status': 'Not implemented; IDs are not vehicle counts.'}

VARIANT_BURST_FRAMES = {}
VARIANT_BURST_RECORDS = {}
SELECTED_FRAME_ARTIFACTS = {}

with stage('full_independent_cached_off_on_sequences'):
    for variant in VARIANTS:
        variant_dir = RUN_DIR / variant
        variant_dir.mkdir(); (variant_dir/'samples').mkdir(); (variant_dir/'temporal').mkdir()
        execution = {'status': 'initialized', 'frames_processed': 0, 'configuration': VARIANT_CONFIGS[variant]}
        REPORT['variants'][variant] = execution
        BaseTrack.reset_id()  # Each independent video/variant begins a fresh sequence.
        if variant == 'reid_on':
            with profiled_ort_session_creation(variant_dir/'ort_encoder_profile'):
                tracker = AppearanceEvidenceBOTSORT(IterableSimpleNamespace(**VARIANT_CONFIGS[variant]))
            actual_encoder = tracker.encoder_evidence.encoder
            session = actual_encoder.model.session
            providers = session.get_providers()
            require('CUDAExecutionProvider' in providers and providers[0] == 'CUDAExecutionProvider',
                    'Upstream backend selected CPU instead of requested CUDA; execution is stopped.')
            require(actual_encoder.device.type == 'cuda' and actual_encoder.model.device.type == 'cuda',
                    'Encoder/backend actual device is not CUDA.')
            require(not actual_encoder.is_pt and actual_encoder.imgsz == (224, 224), 'Unexpected encoder path/crop size.')
            require(not actual_encoder.fp16 and actual_encoder.batch_size is None, 'Expected dynamic float32 ONNX candidate.')
            execution['encoder_runtime'] = {'class': type(actual_encoder).__name__, 'providers': providers,
                'provider_options': session.get_provider_options(), 'ort_version': ort.__version__,
                'device': str(actual_encoder.device), 'backend_device': str(actual_encoder.model.device),
                'imgsz': list(actual_encoder.imgsz), 'fp16': actual_encoder.fp16, 'static_batch_size': actual_encoder.batch_size,
                'model_inputs': [{'name': i.name, 'shape': i.shape, 'type': i.type} for i in session.get_inputs()],
                'model_outputs': [{'name': i.name, 'shape': i.shape, 'type': i.type} for i in session.get_outputs()],
                'metadata': session.get_modelmeta().custom_metadata_map,
                'no_runtime_fallback': True}
        else:
            tracker = AppearanceEvidenceBOTSORT(IterableSimpleNamespace(**VARIANT_CONFIGS[variant]))
            require(tracker.encoder is None, 'Off variant unexpectedly has appearance encoder.')
        require(tracker.gmc.method is None, 'GMC unexpectedly enabled.')
        registry, states, emitted_ids = {}, {}, set()
        tracker_times, decode_times, encoder_times = [], [], []
        render_seconds = 0.0; emitted_count = 0; empty_count = 0; no_emitted_count = 0
        bursts = {start: [] for start, end in BURSTS}; selected = []; burst_records = []
        cap = cv2.VideoCapture(str(VIDEO_PATH))
        require(cap.isOpened(), 'Matching source video failed to open.')
        require(int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) == WIDTH and int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) == HEIGHT,
                'Actual decoded dimensions do not match cache.')
        execution['decode_metadata'] = {'width': cap.get(cv2.CAP_PROP_FRAME_WIDTH), 'height': cap.get(cv2.CAP_PROP_FRAME_HEIGHT),
                                        'fps': cap.get(cv2.CAP_PROP_FPS), 'frame_count': cap.get(cv2.CAP_PROP_FRAME_COUNT),
                                        'backend': cap.getBackendName()}
        loop_started = time.perf_counter()
        try:
            with (variant_dir/'track_records.jsonl').open('w') as out, (variant_dir/'appearance_records.jsonl').open('w') as appearance_out:
                for row in CACHE_ROWS:
                    index = row['frame_index']; started = time.perf_counter()
                    ok, frame = cap.read(); decode_times.append(time.perf_counter()-started)
                    require(ok and frame.shape[:2] == (HEIGHT, WIDTH), f'Decoder failed alignment at frame {index}.')
                    tracker.cost_calls = []
                    if tracker.encoder_evidence is not None: tracker.encoder_evidence.calls = []
                    started = time.perf_counter()
                    result = tracker.update(detection_boxes(row), img=frame if variant == 'reid_on' else None)
                    tracker_times.append(time.perf_counter()-started)
                    emitted = validate_emitted(result, row)
                    evidence = frame_evidence(tracker, emitted, row, registry, states)
                    evidence.update(variant=variant, tracker_update_wall_seconds=tracker_times[-1],
                                    timestamp_current_decoder_backend_ms=float(cap.get(cv2.CAP_PROP_POS_MSEC)))
                    encoder_calls = tracker.encoder_evidence.calls if tracker.encoder_evidence is not None else []
                    encoder_seconds = sum(call['encoder_wall_seconds'] for call in encoder_calls)
                    encoder_times.append(encoder_seconds)
                    if variant == 'reid_on' and encoder_calls and 'cuda_execution_evidence' not in execution:
                        execution['cuda_execution_evidence'] = cuda_profile_evidence(session, variant_dir)
                        execution['cuda_execution_evidence']['profiled_source_frame'] = index
                        execution['cuda_execution_evidence']['scope'] = 'First actual encoder-bearing update of this full sequence; profiling stopped immediately to avoid full-run diagnostic overhead.'
                    appearance_record = {'variant': variant, 'frame_index': index,
                                         'encoder_calls': encoder_calls, 'association_calls': tracker.cost_calls,
                                         'encoder_wall_seconds': encoder_seconds}
                    out.write(json.dumps(evidence, allow_nan=False)+'\n')
                    appearance_out.write(json.dumps(appearance_record, allow_nan=False)+'\n')
                    emitted_ids.update(obs['track_id'] for obs in evidence['emitted_tracks'])
                    emitted_count += len(evidence['emitted_tracks']); empty_count += int(not row['detections'])
                    no_emitted_count += int(not evidence['emitted_tracks'])
                    matched_burst = next((start for start, end in BURSTS if start <= index <= end), None)
                    if matched_burst is not None or index in REPRESENTATIVE_INDICES or index in FOCUS_INDICES:
                        started = time.perf_counter(); annotated = render_tracks(frame, evidence)
                        if matched_burst is not None:
                            bursts[matched_burst].append((index, annotated)); burst_records.append(evidence)
                        if index in REPRESENTATIVE_INDICES or index in FOCUS_INDICES:
                            raw_path = variant_dir/'samples'/f'frame_{index:09d}_raw.png'
                            tracked_path = variant_dir/'samples'/f'frame_{index:09d}_tracked.png'
                            require(cv2.imwrite(str(raw_path), frame) and cv2.imwrite(str(tracked_path), annotated), 'PNG write failed.')
                            selected.append({'frame_index': index, 'raw': str(raw_path.relative_to(RUN_DIR)),
                                             'raw_sha256': sha256_file(raw_path), 'annotated': str(tracked_path.relative_to(RUN_DIR)),
                                             'annotated_sha256': sha256_file(tracked_path)})
                        render_seconds += time.perf_counter()-started
                    execution['frames_processed'] = index+1
                    REPORT['frames_processed'] = sum(v['frames_processed'] for v in REPORT['variants'].values())
                    if (index+1) % 1000 == 0:
                        out.flush(); appearance_out.flush(); write_report()
                        print(f'{variant}: {index+1}/{len(CACHE_ROWS)} cached frames; ID totals are not vehicle totals.', flush=True)
                started = time.perf_counter(); extra_ok, _ = cap.read(); decode_times.append(time.perf_counter()-started)
                require(not extra_ok, 'Decoded extra frame beyond validated cache.')
                execution['decode_stop'] = 'opencv_read_false_eof_or_decode_failure_unknown_after_expected_rows'
        finally:
            cap.release()
            execution['timing'] = {'loop_wall_seconds': time.perf_counter()-loop_started,
                'tracker_update_calls': len(tracker_times), 'tracker_update_wall_seconds': sum(tracker_times),
                'tracker_update_wall_ms_mean': statistics.mean(tracker_times)*1000 if tracker_times else None,
                'tracker_update_wall_ms_median': statistics.median(tracker_times)*1000 if tracker_times else None,
                'encoder_wall_seconds_inside_tracker': sum(encoder_times),
                'decoder_read_calls_including_final_probe': len(decode_times), 'decode_read_wall_seconds': sum(decode_times),
                'selected_render_and_png_write_wall_seconds': render_seconds,
                'limits': 'Instrumented tracker includes adapter, association and cost validation; on variant also actual crop/encoder inference. Encoder is a subset, not additional time. Loop also includes JSON serialization/IO. Sequential order/warmup may affect runtime.'}
            execution['association_mechanism_diagnostics'] = dict(tracker.cost_totals)
            if tracker.encoder_evidence is not None:
                execution['embedding_diagnostics'] = {'encoder_calls': tracker.encoder_evidence.total_calls,
                    'finite_nonzero_features_produced': tracker.encoder_evidence.total_features,
                    'embedding_dimension': 512, 'encoder_observer_wall_seconds': tracker.encoder_evidence.total_wall_seconds}
            (variant_dir/'track_summary.json').write_text(json.dumps({
                'video_sha256': EXPECTED_VIDEO_SHA256, 'run_id': RUN_ID, 'variant': variant,
                'track_id_scope': 'one independent video tracker/variant',
                'summary_semantics': 'All captured IDs including unconfirmed; not physical-object/traffic totals.',
                'tracks': list(registry.values())}, indent=2, allow_nan=False)+'\n')
            execution['observed_process_peak_rss_bytes'] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss*1024
        require(execution['frames_processed'] == EXPECTED_FRAME_COUNT, 'Variant did not consume full cache.')
        for entry in registry.values():
            require(entry['detection_count'] == len(entry['trajectory']), 'History/count bookkeeping mismatch.')
            indices = [obs['frame_index'] for obs in entry['trajectory']]
            require(indices == sorted(set(indices)), 'Nonmonotonic per-track history.')
            require(entry['direction'] is None and entry['counting_state'] is None, 'Later-phase state introduced.')
        execution['diagnostics'] = variant_diagnostics(registry, len(CACHE_ROWS), emitted_ids, emitted_count, empty_count, no_emitted_count)
        if variant == 'reid_on':
            require(tracker.encoder_evidence.total_features > 0 and tracker.cost_totals['nonempty_feature_association_calls'] > 0,
                    'No actual appearance features reached association.')
            require(execution.get('cuda_execution_evidence', {}).get('actual_cuda_execution_confirmed'),
                    'Actual ONNX CUDA execution was not retained.')
            REPORT['environment']['reid_inference_executed'] = True
        execution['status'] = 'completed_full_cached_sequence_review_pending'
        VARIANT_BURST_FRAMES[variant] = bursts; VARIANT_BURST_RECORDS[variant] = burst_records
        SELECTED_FRAME_ARTIFACTS[variant] = selected
        del tracker, registry, states
        write_report()
    off_samples = {entry['frame_index']: entry['raw_sha256'] for entry in SELECTED_FRAME_ARTIFACTS['reid_off']}
    on_samples = {entry['frame_index']: entry['raw_sha256'] for entry in SELECTED_FRAME_ARTIFACTS['reid_on']}
    require(off_samples == on_samples, 'Selected native source pixels differ between decoder passes.')
    REPORT['paired_native_source_pixels'] = {'selected_frame_sha256_equal': True, 'frames_checked': sorted(off_samples),
        'limits': 'Selected native PNG byte hashes match; video/cache hashes and ordered full decoding provide input identity. No source PTS/decode integrity guarantee.'}
    REPORT['status'] = 'completed_controlled_reid_sequences_manual_review_pending'


In [ ]:
def make_contact_sheet(frames, title, path, indices=None, crop=None):
    if indices is not None:
        frames = [frames[index] for index in indices]
    columns = 2
    tile_width = 640
    tile_height = 360 if crop is None else round(tile_width*(crop[3]-crop[1])/(crop[2]-crop[0]))
    rows = math.ceil(len(frames)/columns)
    sheet = Image.new('RGB', (columns*tile_width, 40+rows*(tile_height+24)), 'black')
    draw = ImageDraw.Draw(sheet)
    draw.text((10, 10), title, fill='white')
    for position, (frame_index, bgr) in enumerate(frames):
        if crop is not None:
            x1,y1,x2,y2 = crop; bgr = bgr[y1:y2, x1:x2]
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tile = Image.fromarray(rgb).resize((tile_width, tile_height), Image.Resampling.LANCZOS)
        x = (position%columns)*tile_width; y = 40+(position//columns)*(tile_height+24)
        draw.text((x+8, y+4), f'frame {frame_index}', fill='white')
        sheet.paste(tile, (x, y+24))
    sheet.save(path)


def make_paired_contact_sheet(off_frames, on_frames, title, path, positions=None, crop=None):
    if positions is not None:
        off_frames = [off_frames[i] for i in positions]; on_frames = [on_frames[i] for i in positions]
    require([i for i, _ in off_frames] == [i for i, _ in on_frames], 'Paired contact sheet source indices mismatch.')
    width = 640; height = 360 if crop is None else round(width*(crop[3]-crop[1])/(crop[2]-crop[0]))
    sheet = Image.new('RGB', (width*2, 40+len(off_frames)*(height+24)), 'black')
    draw = ImageDraw.Draw(sheet); draw.text((10, 10), title+' | left OFF, right ON', fill='white')
    for row_index, pair in enumerate(zip(off_frames, on_frames)):
        for column, (index, image) in enumerate(pair):
            if crop is not None:
                x1,y1,x2,y2=crop; image=image[y1:y2, x1:x2]
            tile = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).resize((width,height), Image.Resampling.LANCZOS)
            x=column*width; y=40+row_index*(height+24)
            draw.text((x+8,y+4), f'frame {index} | {VARIANTS[column]}', fill='white'); sheet.paste(tile,(x,y+24))
    sheet.save(path)

with stage('paired_temporal_export_and_serialized_consistency_audit'):
    artifacts = {'variants': {}, 'paired_contact_sheets': []}
    for variant in VARIANTS:
        variant_dir=RUN_DIR/variant; temporal=[]
        for start,end in BURSTS:
            frames=VARIANT_BURST_FRAMES[variant][start]
            require([i for i,_ in frames] == list(range(start,end+1)), 'Incomplete temporal burst.')
            clip=variant_dir/'temporal'/f'burst_{start:09d}_{end:09d}_tracked.mp4'
            writer=cv2.VideoWriter(str(clip),cv2.VideoWriter_fourcc(*'mp4v'),FPS,(WIDTH,HEIGHT))
            require(writer.isOpened(),'Actual temporal MP4 writer unavailable; no codec substitution.')
            try:
                for _,image in frames: writer.write(image)
            finally: writer.release()
            probe=cv2.VideoCapture(str(clip)); count=0
            try:
                while True:
                    ok,_=probe.read()
                    if not ok: break
                    count+=1
            finally: probe.release()
            require(count == len(frames),'Retained temporal clip frame count mismatch.')
            sheet=variant_dir/'temporal'/f'burst_{start:09d}_{end:09d}_contact_sheet.png'
            make_contact_sheet(frames,f'{variant} actual full-history evidence {start}-{end}',sheet,indices=[0,5,10,15,20,29])
            temporal.append({'start_frame':start,'end_frame':end,'frame_count':count,
                             'clip':str(clip.relative_to(RUN_DIR)),'clip_sha256':sha256_file(clip),
                             'contact_sheet':str(sheet.relative_to(RUN_DIR)),'contact_sheet_sha256':sha256_file(sheet)})
        representative=variant_dir/'representative_temporal_track_records.json'
        representative.write_text(json.dumps(VARIANT_BURST_RECORDS[variant],indent=2,allow_nan=False)+'\n')
        serialized_rows=0
        with (variant_dir/'track_records.jsonl').open() as handle:
            for index,line in enumerate(handle):
                record=json.loads(line)
                require(record['frame_index']==index and record['variant']==variant and record['video_sha256']==EXPECTED_VIDEO_SHA256,
                        'Serialized frame/variant identity mismatch.')
                require(record['raw_detections']==CACHE_ROWS[index]['detections'],'Serialized raw detection evidence changed.')
                ids=[obs['track_id'] for obs in record['emitted_tracks']]
                require(len(ids)==len(set(ids)),'Serialized duplicate emitted IDs.')
                serialized_rows+=1
        require(serialized_rows==EXPECTED_FRAME_COUNT,'Serialized row count mismatch.')
        artifacts['variants'][variant]={'track_records':str((variant_dir/'track_records.jsonl').relative_to(RUN_DIR)),
            'track_records_sha256':sha256_file(variant_dir/'track_records.jsonl'),
            'track_summary':str((variant_dir/'track_summary.json').relative_to(RUN_DIR)),
            'track_summary_sha256':sha256_file(variant_dir/'track_summary.json'),
            'appearance_records':str((variant_dir/'appearance_records.jsonl').relative_to(RUN_DIR)),
            'appearance_records_sha256':sha256_file(variant_dir/'appearance_records.jsonl'),
            'representative_temporal_track_records':str(representative.relative_to(RUN_DIR)),
            'representative_temporal_track_records_sha256':sha256_file(representative),
            'sample_frames':SELECTED_FRAME_ARTIFACTS[variant],'temporal_bursts':temporal}
        REPORT['variants'][variant]['serialized_consistency_audit']={'status':'passed','ordered_rows':serialized_rows,
            'raw_detection_preservation':True,'unique_emitted_ids':True,'ordered_histories':True,'temporal_clips_decode':True}
    for start,end in BURSTS:
        sheet=RUN_DIR/'temporal'/f'paired_burst_{start:09d}_{end:09d}_contact_sheet.png'
        make_paired_contact_sheet(VARIANT_BURST_FRAMES['reid_off'][start],VARIANT_BURST_FRAMES['reid_on'][start],
                                 f'Actual paired full-history {start}-{end}',sheet,positions=[0,5,10,15,20,29])
        artifacts['paired_contact_sheets'].append({'scope':[start,end],'path':str(sheet.relative_to(RUN_DIR)),'sha256':sha256_file(sheet)})
    for name,indices,start,crop in [('entering_scooter',FOCUS_INDICES,7696,FOCUS_CROP_XYXY),
                                    ('red_scooter',RED_FOCUS_INDICES,5125,RED_FOCUS_CROP_XYXY)]:
        off=[(i,im) for i,im in VARIANT_BURST_FRAMES['reid_off'][start] if i in indices]
        on=[(i,im) for i,im in VARIANT_BURST_FRAMES['reid_on'][start] if i in indices]
        require([i for i,_ in off]==indices and [i for i,_ in on]==indices,'Focused paired evidence incomplete.')
        sheet=RUN_DIR/'temporal'/f'paired_{name}_{indices[0]:09d}_{indices[-1]:09d}_contact_sheet.png'
        make_paired_contact_sheet(off,on,f'{name}; display crop only; actual off/on IDs',sheet,crop=crop)
        artifacts['paired_contact_sheets'].append({'scope':indices,'display_crop_xyxy':list(crop),
                                                  'path':str(sheet.relative_to(RUN_DIR)),'sha256':sha256_file(sheet)})
    REPORT['artifacts']=artifacts
    REPORT['baseline_compatibility_with_original_phase3']={'status':'pending_independent_retained_artifact_audit',
        'requirement':'Compare canonical emitted observations/lifecycle with original Phase 3; exclude new timing/variant fields.'}
    REPORT['candidate_decision']={'status':'pending_actual_temporal_review','final_vehicle_reid_configuration_accepted':False,
                                 'architecture_reid_removed':False}
    REPORT['limitations']=[
        'No ground truth or physical-identity benchmark; no accuracy/IDF1/MOTA/counting metric or general improvement claim.',
        'Person-domain encoder vehicle suitability unknown; exact upstream crop/channel behavior is retained without custom correction.',
        'Cached detections excluded scores below0.25; missing/lower-confidence observations cannot be restored by ReID.',
        'Class-agnostic association/raw-class changes and rider/pedestrian/product/duplicate policy remain unresolved.',
        'Software counter and appearance-cost differences are mechanism diagnostics, not confirmed identity improvements.',
        'Five temporal bursts plus focused sheets do not establish whole-video identity correctness; subsequent visual review is separate.',
        'Sequential independent passes decode the same hashed source; order/warmup and diagnostic overhead limit runtime comparisons.',
        'No detector inference, GMC/stabilization, night preprocessing, geometry, crossing/counting or Phase5 implementation.',
        'Original execution review fields remain pending; subsequent actual AI-assisted review must be retained separately.']
    REPORT['finished_utc']=datetime.now(timezone.utc).isoformat()
    write_report()
    print(json.dumps({v:REPORT['variants'][v]['diagnostics'] for v in VARIANTS},indent=2))
    print('Actual controlled evidence retained:',RUN_DIR)
write_report()
(RUN_DIR/'run_manifest.json').write_text(json.dumps(REPORT,indent=2,allow_nan=False)+'\n')


## Review after execution

Use paired temporal pixels and linked records to review the red-scooter duplicate, entering rider/motorcycle component handoff, roadside gap/recovery and previously stable cases. Inspect improvements, unchanged cases and regressions. Retain original exports unchanged and add an explicit separate review supplement with provenance, inspected frame scope and candidate decision.

The next gate is an assessment only: Phase 5 concerns deterministic geometry, first on synthetic trajectories. Geometry anchor/boundaries/gap/noise policy must be documented before implementation. ReID remaining part of the architecture does not imply acceptance of this candidate. Current identity/category/duplicate limitations still block trustworthy counting. No Phase 5 code is included here.
